# Preprocessing & First Model

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import min, max
from functools import reduce
import glob
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from IPython.display import display
from pyspark.ml.feature import Imputer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.storagelevel import StorageLevel
from pyspark.ml.classification import GBTClassifier

import os
job_scratch = os.environ["TMPDIR"]  # /scratch/lcheng7/job_#
spark_tmp = os.path.join(job_scratch, "spark_tmp")
os.makedirs(spark_tmp, exist_ok=True)

# Change on creation
TOTAL_CORES = 16
TOTAL_MEM_GB = 200
DRIVER_MEM_GB = 2 # Fixed

EXEC_INSTANCES = TOTAL_CORES - 1
EXEC_MEM_GB = int((TOTAL_MEM_GB - DRIVER_MEM_GB) / EXEC_INSTANCES)

spark = (
    SparkSession.builder
    .appName("fhvhv-model")
    .config("spark.driver.memory", f"{DRIVER_MEM_GB}g")
    .config("spark.executor.instances", str(EXEC_INSTANCES))
    .config("spark.executor.memory", f"{EXEC_MEM_GB}g")
    .config("spark.sql.shuffle.partitions", str(EXEC_INSTANCES * 3))
    .config("spark.default.parallelism", str(EXEC_INSTANCES * 3))
    .getOrCreate()
)

print("TMPDIR =", job_scratch)
print("spark.local.dir =", spark.sparkContext.getConf().get("spark.local.dir"))
print("parallelism =", spark.sparkContext.defaultParallelism)
print("executors:", EXEC_INSTANCES, "executor_mem_gb:", EXEC_MEM_GB)

Matplotlib created a temporary cache directory at /scratch/lcheng7/job_47000704/matplotlib-tqhex88d because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


TMPDIR = /scratch/lcheng7/job_47000704
spark.local.dir = None
parallelism = 45
executors: 15 executor_mem_gb: 13


In [2]:
print("MASTER =", spark.sparkContext.master)
print("DRIVER MEM =", spark.sparkContext.getConf().get("spark.driver.memory"))
print("EXEC INSTANCES =", spark.sparkContext.getConf().get("spark.executor.instances"))

MASTER = local[*]
DRIVER MEM = 2g
EXEC INSTANCES = 15


In [3]:
df_clean = spark.read.parquet("/home/lcheng7/cleaned_dataset/fhvhv_dedup")
df = df_clean

df = df.drop(
    "originating_base_num",
    "on_scene_datetime",
    "dispatching_base_num",
    "request_datetime"
)

In [4]:
df_small = df.limit(5).toPandas()
df_small

,hvfhs_license_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,2022-09-16 13:02:34,2022-09-16 14:18:10,3,1,28.72,4536,101.50,28.58,3.98,0.0,0.0,2.5,0.00,78.13,N,N,,N,N
1,HV0003,2021-07-11 12:24:13,2021-07-11 13:06:10,4,1,13.70,2517,48.38,21.00,2.16,0.0,0.0,2.5,11.08,39.95,N,N,,N,N
2,HV0003,2022-03-13 10:09:12,2022-03-13 10:42:26,4,1,14.34,1994,50.64,21.00,2.22,0.0,0.0,2.5,0.00,39.38,N,N,,N,N
3,HV0003,2019-06-27 14:37:12,2019-06-27 15:46:35,4,1,14.41,4163,74.80,21.00,0.00,0.0,0.0,NaN,0.00,77.62,N,N,,N,
4,HV0003,2022-06-12 12:56:31,2022-06-12 13:34:47,4,1,14.42,2296,58.49,21.00,2.46,0.0,0.0,2.5,0.00,42.73,N,N,,N,N


## Imputing

Missing values were concentrated in policy-related fields such as airport_fee and congestion_surcharge.

For this dataset, a missing airport fee logically means the trip was not an airport trip, so it is reasonable to treat missing values as zero.

Similarly, missing congestion surcharge values likely indicate that the surcharge did not apply to that trip.

Rather than dropping millions of rows, we impute these values to preserve data volume and avoid introducing bias.

In [5]:
df = df.fillna({
    "airport_fee": 0,
    "congestion_surcharge": 0
})

df = df.withColumn(
    "wav_match_flag",
    F.when(
        F.col("wav_match_flag").isNull() |
        (F.length(F.trim(F.col("wav_match_flag").cast("string"))) == 0),
        F.lit("Unknown")
    ).otherwise(F.trim(F.col("wav_match_flag").cast("string")))
)

df.groupBy("wav_match_flag").count().show()

+--------------+---------+
|wav_match_flag|    count|
+--------------+---------+
|             Y| 26977884|
|             N|607638261|
|       Unknown|110657176|
+--------------+---------+



In [6]:
null_summary = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_summary.toPandas().T

,0
hvfhs_license_num,0
pickup_datetime,0
dropoff_datetime,0
PULocationID,0
DOLocationID,0
trip_miles,0
trip_time,0
base_passenger_fare,0
tolls,0
bcf,0


## Scaling/Transforming

In [7]:
# target (tip)
df1 = df.withColumn("tip_binary", F.when(F.col("tips").cast("double") > 0, 1).otherwise(0))

# numeric sanity casts first
df1 = (
    df1
    .withColumn("base_passenger_fare", F.col("base_passenger_fare").cast("double"))
    .withColumn("trip_miles", F.col("trip_miles").cast("double"))
    .withColumn("trip_time", F.col("trip_time").cast("double"))
    .withColumn("tolls", F.col("tolls").cast("double"))
    .withColumn("bcf", F.col("bcf").cast("double"))
    .withColumn("sales_tax", F.col("sales_tax").cast("double"))
    .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
    .withColumn("airport_fee", F.col("airport_fee").cast("double"))
    .withColumn("tips", F.col("tips").cast("double"))
)

# time features
df1 = (
    df1
    .withColumn("trip_minutes", F.col("trip_time") / F.lit(60.0))
    .withColumn("pickup_hour", F.hour("pickup_datetime").cast("double"))
    .withColumn("pickup_dow", F.dayofweek("pickup_datetime").cast("double"))
    .withColumn("pickup_month", F.month("pickup_datetime").cast("double"))
    .withColumn("log_fare", F.when(F.col("base_passenger_fare") >= 0, F.log1p("base_passenger_fare")).otherwise(None))
    .withColumn("log_miles", F.when(F.col("trip_miles") >= 0, F.log1p("trip_miles")).otherwise(None))
    .withColumn("log_minutes", F.when(F.col("trip_minutes") >= 0, F.log1p("trip_minutes")).otherwise(None))
)

# flags
df1 = (
    df1
    .withColumn("is_weekend", F.when(F.dayofweek("pickup_datetime").isin([1, 7]), 1.0).otherwise(0.0))
    .withColumn("is_night", F.when(F.hour("pickup_datetime").isin(list(range(0,6)) + list(range(20,24))), 1.0).otherwise(0.0))
    .withColumn("has_airport_fee", F.when(F.col("airport_fee") > 0, 1.0).otherwise(0.0))
    .withColumn("has_congestion_fee", F.when(F.col("congestion_surcharge") > 0, 1.0).otherwise(0.0))
)

df1 = (
    df1
    .withColumn(
        "total_surcharges",
        F.coalesce("tolls", F.lit(0.0)) +
        F.coalesce("bcf", F.lit(0.0)) +
        F.coalesce("sales_tax", F.lit(0.0)) +
        F.coalesce("congestion_surcharge", F.lit(0.0)) +
        F.coalesce("airport_fee", F.lit(0.0))
    )
)

df1 = (
    df1
    .withColumn(
        "fare_per_mile",
        F.when(F.col("trip_miles") > 0, F.col("base_passenger_fare") / F.col("trip_miles")).otherwise(None)
    )
)

df1.select(
    "tip_binary", "log_fare", "log_miles", "log_minutes", "pickup_hour", 
    "pickup_dow", "pickup_month","is_weekend","is_night",
    "has_airport_fee","has_congestion_fee",
    "total_surcharges", "fare_per_mile"
).limit(5).toPandas()

,tip_binary,log_fare,log_miles,log_minutes,pickup_hour,pickup_dow,pickup_month,is_weekend,is_night,has_airport_fee,has_congestion_fee,total_surcharges,fare_per_mile
0,0,4.629863,3.391820,4.338597,13.0,6.0,9.0,0.0,0.0,1.0,0.0,35.06,3.534123
1,1,3.899545,2.687847,3.760037,12.0,1.0,7.0,1.0,0.0,1.0,0.0,25.66,3.531387
2,0,3.944297,2.730464,3.533200,10.0,1.0,3.0,1.0,0.0,1.0,0.0,25.72,3.531381
3,0,4.328098,2.735017,4.253956,14.0,5.0,6.0,0.0,0.0,0.0,0.0,21.00,5.190840
4,0,4.085808,2.735665,3.670376,12.0,1.0,6.0,1.0,0.0,1.0,0.0,25.96,4.056172


In [ ]:
num_cols = [
    "log_fare",
    "log_miles",
    "log_minutes",
    "tolls",
    "congestion_surcharge",
    "airport_fee",
    "pickup_hour",
    "pickup_dow",
    "pickup_month",
    "is_weekend",
    "is_night",
    "has_airport_fee",
    "has_congestion_fee",
    "total_surcharges",
    "fare_per_mile"
]

# impute new nulls
df_num = df1
for c in num_cols:
    df_num = df_num.withColumn(
        c,
        F.when(F.isnan(F.col(c)), None).otherwise(F.col(c)).cast("double")
    )

imputer = Imputer(
    inputCols=num_cols,
    outputCols=[f"{c}_imp" for c in num_cols],
    strategy="median"
)
imputer_model = imputer.fit(df_num)
df_imp = imputer_model.transform(df_num)

# Assemble

assembler_num = VectorAssembler(
    inputCols=[f"{c}_imp" for c in num_cols],
    outputCol="num_features",
    handleInvalid="error"
)
df2 = assembler_num.transform(df_imp)

scaler = StandardScaler(
    inputCol="num_features",
    outputCol="num_scaled",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)

df3.select("num_features", "num_scaled", "tip_binary").limit(5).toPandas()

In [ ]:
print("Rows:", df3.count())
df3.groupBy("tip_binary").count().show()

## Encoding

In [ ]:
# Categorical

cat_cols = [
    "hvfhs_license_num",
    "shared_request_flag",
    "shared_match_flag",
    "access_a_ride_flag",
    "wav_request_flag",
    "wav_match_flag"
]

df5 = df3

for c in cat_cols:
    df5 = df5.withColumn(c, F.trim(F.col(c).cast("string")))
    df5 = df5.withColumn(
        c,
        F.when(F.col(c).isNull() | (F.length(F.col(c)) == 0), F.lit("Unknown")).otherwise(F.col(c))
    )

for c in cat_cols:
    print(c)
    pdf = df5.groupBy(c).count().orderBy(F.desc("count")).limit(10).toPandas()
    display(pdf)

In [ ]:
# StringIndex + OneHot Encode

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

df_idx = df5
for idx in indexers:
    df_idx = idx.fit(df_idx).transform(df_idx)

df_enc = encoder.fit(df_idx).transform(df_idx)

In [ ]:
# Check new dataset
df_enc.select(
    cat_cols +
    [f"{c}_idx" for c in cat_cols] +
    [f"{c}_ohe" for c in cat_cols]
).limit(3).toPandas().T

## Feature Engineering

In [ ]:
assembler_all = VectorAssembler(
    inputCols=["num_scaled"] + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep"
)

df_final = assembler_all.transform(df_enc).select(
    F.col("tip_binary").cast("int").alias("label"),
    "features"
)

df_final.limit(5).toPandas()

# Modeling

In [ ]:
df_model = df_final.select("label", "features")
df_sample = df_model.sample(withReplacement=False, fraction=0.01, seed=42)
df_sample.groupBy("label").count().show()

In [ ]:
df_sample = df_sample.repartition(96)
# df_final = df_final.repartition(96)

train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.persist(StorageLevel.DISK_ONLY)
test_df  = test_df.persist(StorageLevel.DISK_ONLY)

train_n = train_df.count()
test_n  = test_df.count()

In [ ]:
print("train rows =", train_n)
print("test rows  =", test_n)

## Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=10,
    maxDepth=6,
    maxBins=16,
    subsamplingRate=0.7,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_df)

In [ ]:
pred_test = rf_model.transform(test_df)

In [ ]:
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)

roc_auc = evaluator_roc.evaluate(pred_test)
pr_auc = evaluator_pr.evaluate(pred_test)

print("TEST ROC-AUC:", roc_auc)
print("TEST PR-AUC :", pr_auc)

## GBTClassifier

In [ ]:
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxIter=20,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.7,
    seed=42
)

gbt_model = gbt.fit(train_df)

In [ ]:
pred_test = gbt_model.transform(test_df)

In [ ]:
roc_auc = evaluator_roc.evaluate(pred_test)
pr_auc = evaluator_pr.evaluate(pred_test)

print("TEST ROC-AUC:", roc_auc)
print("TEST PR-AUC :", pr_auc)

In [ ]:
# if needed

# try:
#     spark.stop()
# except:
#     pass